In [0]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 103.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

%md
The schema must include: name, car_number, experience, rating, and timestamp...  a new Delta table called bronze.drivers... store the checkpoint files in your raw_files Volume
...Use the Python Faker library to generate a continuous mock data stream for drivers.

Later i did "Update your Faker script to generate an additional phone column."


In [0]:
from datetime import datetime, timezone
import pandas as pd
from faker import Faker

fake = Faker()

# Target Delta table and checkpoint location (stored in the raw_files Volume)
TARGET_TABLE = "aaa.bronze.drivers"
CHECKPOINT_DIR = "/Volumes/aaa/raw_files/data/checkpoints/drivers_stream"

# Schema of the mock driver records
DRIVER_SCHEMA = "name string, car_number string, experience int, rating double, phone string, timestamp timestamp"
DRIVER_FIELDS = [c.split()[0] for c in DRIVER_SCHEMA.split(",")]


def generate_driver_rows(batches):
    """Converts each rate-source batch into an equal number of Faker driver records."""
    for batch in batches:
        records = []
        for _ in range(len(batch)):
            records.append({
                "name": fake.name(),
                "car_number": fake.license_plate(),
                "experience": fake.random_int(min=0, max=30),
                "rating": round(fake.pyfloat(min_value=1.0, max_value=5.0), 2),
                "phone": fake.numerify('###-###-####'),
                "timestamp": datetime.now(timezone.utc),
            })
        yield pd.DataFrame(records, columns=DRIVER_FIELDS)


print("Faker generator ready for fields:", DRIVER_FIELDS)

Faker generator ready for fields: ['name', 'car_number', 'experience', 'rating', 'phone', 'timestamp']


Write this stream into a new Delta table called bronze.drivers. Enable checkpointing and store the checkpoint files in your raw_files Volume. 

In [0]:
# Continuous rate source feeding the Faker generator
rate_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load())

drivers_stream = rate_df.mapInPandas(generate_driver_rows, schema=DRIVER_SCHEMA)

drivers_query = (drivers_stream.writeStream
    .format("delta")
    .outputMode("append")
    .queryName("drivers_to_bronze")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .option("mergeSchema", "true")
    .toTable(TARGET_TABLE))

print(f"Query '{drivers_query.name}' -> {drivers_query.status['message']}")
print(f"Target table: {TARGET_TABLE}")
print(f"Checkpoint location: {CHECKPOINT_DIR}")

Query 'drivers_to_bronze' -> Initializing sources
Target table: aaa.bronze.drivers
Checkpoint location: /Volumes/aaa/raw_files/data/checkpoints/drivers_stream


Verify the records are actively ingesting

In [0]:
import time

def ingestion_snapshot():
    count = spark.table(TARGET_TABLE).count()
    print(f"query '{drivers_query.name}' active={drivers_query.isActive} | rows in {TARGET_TABLE}: {count}")
    return count


before = ingestion_snapshot()
time.sleep(15)
after = ingestion_snapshot()

print(f"\nIngestion growth over 15s: +{after - before} rows")
display(spark.sql(f"SELECT * FROM {TARGET_TABLE} ORDER BY timestamp DESC LIMIT 5"))

query 'drivers_to_bronze' active=True | rows in aaa.bronze.drivers: 208
query 'drivers_to_bronze' active=True | rows in aaa.bronze.drivers: 3101

Ingestion growth over 15s: +2893 rows


name,car_number,experience,rating,timestamp,phone
Teresa Brooks,V 744195,7,4.91,2026-11-03T14:41:35.297376Z,694.982.4875
Barbara Jacobs,804 WHQ,20,2.88,2026-11-03T14:41:35.295964Z,001-405-655-6544
Teresa Brooks,V 744195,7,4.91,2026-11-03T14:41:35.292808Z,694.982.4875
Barbara Jacobs,804 WHQ,20,2.88,2026-11-03T14:41:35.292561Z,001-405-655-6544
Teresa Brooks,V 744195,7,4.91,2026-11-03T14:41:35.279789Z,694.982.4875


Stop the stream

In [0]:
# Stop the current streaming query before restarting with the evolved schema
for q in spark.streams.active:
    print("stopping:", q.name)
    q.stop()

print("active streams remaining:", len(spark.streams.active))

stopping: drivers_to_bronze
active streams remaining: 0


In [0]:
# Confirm the phone column landed in the table schema and new records carry it
print("Table schema after evolution:")
spark.table(TARGET_TABLE).printSchema()

evolution_check = spark.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT_IF(phone IS NOT NULL) AS rows_with_phone,
           COUNT_IF(phone IS NULL) AS rows_without_phone
    FROM {TARGET_TABLE}
""")
display(evolution_check)

Table schema after evolution:
root
 |-- name: string (nullable = true)
 |-- car_number: string (nullable = true)
 |-- experience: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- phone: string (nullable = true)



total_rows,rows_with_phone,rows_without_phone
3154,2946,208


name,car_number,phone,rating,timestamp
Teresa Brooks,V 744195,694.982.4875,4.91,2026-11-03T14:41:43.712152Z
Eric Edwards,JVG-5547,973-498-9312x84463,2.0,2026-11-03T14:41:43.712007Z
Barbara Jacobs,804 WHQ,001-405-655-6544,2.88,2026-11-03T14:41:43.71192Z
Teresa Brooks,V 744195,694.982.4875,4.91,2026-11-03T14:41:43.711835Z
Teresa Brooks,V 744195,694.982.4875,4.91,2026-11-03T14:41:43.711751Z


Query the bronze.drivers table using the VERSION AS OF syntax to view the data exactly as it was before the phone column was added. Finally, use the RESTORE command to permanently roll back the table to that previous version.


In [0]:
history = spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE}").collect()
versions = sorted((r["version"] for r in history), reverse=True)

LAST_PRE_PHONE_VERSION = None
for v in versions:
    columns_then = spark.read.option("versionAsOf", v).table(TARGET_TABLE).columns
    if "phone" not in columns_then:
        LAST_PRE_PHONE_VERSION = v
        break

print(f"latest version before the phone column was added: {LAST_PRE_PHONE_VERSION}")

latest version before the phone column was added: 14


In [0]:
before_phone_df = spark.sql(f"SELECT * FROM {TARGET_TABLE} VERSION AS OF {LAST_PRE_PHONE_VERSION}")

display(spark.sql(f"SELECT COUNT(*) AS rows_before_phone FROM {TARGET_TABLE} VERSION AS OF {LAST_PRE_PHONE_VERSION}"))
display(before_phone_df.orderBy("timestamp").limit(15))

rows_before_phone
208


name,car_number,experience,rating,timestamp
Barbara Jacobs,804 WHQ,20,2.88,2026-11-03T14:31:27.433091Z
Barbara Jacobs,804 WHQ,20,2.88,2026-11-03T14:31:27.440322Z
Barbara Jacobs,804 WHQ,20,2.88,2026-11-03T14:31:27.446452Z
Barbara Jacobs,804 WHQ,20,2.88,2026-11-03T14:31:29.782042Z
Amy Humphrey,6ZLL 54,8,2.99,2026-11-03T14:31:29.78227Z
William Mercer,74G V41,25,3.48,2026-11-03T14:31:29.782499Z
Colleen Matthews,248 XQY,15,3.94,2026-11-03T14:31:29.782711Z
Kyle Hansen,355 RJV,9,1.2,2026-11-03T14:31:29.782964Z
Miss Tracy Rivera,9U 89312,16,3.51,2026-11-03T14:31:29.783147Z
Katherine Brewer,34-0581J,30,1.5,2026-11-03T14:31:29.783315Z


In [0]:
# Permanently roll the table back to that previous version
# Note: the correct syntax is TO VERSION AS OF — a plain TO VERSION <n> is a parse error
restore_result = spark.sql(f"RESTORE TABLE {TARGET_TABLE} TO VERSION AS OF {LAST_PRE_PHONE_VERSION}")
display(restore_result)

# Verify the rollback: schema without phone, row count of the restored snapshot
print("schema after restore:")
spark.table(TARGET_TABLE).printSchema()

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
32933,17,9,17,57396,32933


schema after restore:
root
 |-- name: string (nullable = true)
 |-- car_number: string (nullable = true)
 |-- experience: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



# Data Cleansing

In [0]:
# Data Cleansing - Bronze to Silver
from pyspark.sql import functions as F

# silver.taxi
raw_taxi = spark.table("aaa.bronze.taxi")
silver_taxi = (raw_taxi
    .filter((F.col("passenger_count") > 0) & (F.col("trip_distance") > 0) & (F.col("fare_amount") > 0))
    .withColumn("description", F.when(F.col("payment_type") == 1, "Credit Card")
                                   .when(F.col("payment_type") == 2, "Cash")
                                   .otherwise("Other"))
    .withColumn("ingest_time", F.current_timestamp())
    .dropDuplicates())
silver_taxi.write.mode("overwrite").format("delta").saveAsTable("aaa.silver.taxi")

# silver.drivers
raw_drivers = spark.table("aaa.bronze.drivers")
silver_drivers = (raw_drivers
    .filter((F.col("experience") > 0) & (F.col("rating").between(1.0, 5.0)))
    .withColumn("driver_id", F.sha2(F.concat_ws("|", F.col("name"), F.col("car_number")), 256))
    .withColumn("ingest_time", F.current_timestamp())
    .dropDuplicates(["driver_id"]))
silver_drivers.write.mode("overwrite").format("delta").saveAsTable("aaa.silver.drivers")

# silver.users
raw_users = spark.table("aaa.bronze.users")
date_range_days = 365
silver_users = (
    raw_users
    .filter(F.col("name").isNotNull())
    .withColumn("ingest_time", F.current_timestamp())
    .withColumn(
        "registration_date",
        F.expr(f"date_sub(current_date(), cast(rand() * {date_range_days} as int))")
    )
    .dropDuplicates()
silver_users.write.mode("overwrite").format("delta").saveAsTable("aaa.silver.users")


# Aggregations & Optimizations

In [0]:
# gold.taxi_metrics
from pyspark.sql import functions as F

taxi_metrics = (
    spark.table("aaa.silver.taxi")
    .groupBy("PULocationID", "description")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue")
    )
)
taxi_metrics.write.mode("overwrite").format("delta").saveAsTable("aaa.gold.taxi_metrics")
# i manually chose columns for clustering (those which are used for grouping)
spark.sql("ALTER TABLE aaa.gold.taxi_metrics CLUSTER BY (PULocationID, description)")

# gold.driver_metrics

driver_metrics = (
    spark.table("aaa.silver.drivers")
    .groupBy("experience")
    .agg(
        F.countDistinct("driver_id").alias("drivers_count"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
)
driver_metrics.write.mode("overwrite").format("delta").saveAsTable("aaa.gold.driver_metrics")
# i manually chose the column for z-ordering (the one which is used for grouping)
spark.sql("OPTIMIZE aaa.gold.driver_metrics ZORDER BY (experience)")

# gold.user_metrics

user_metrics = (
    spark.table("aaa.silver.users")
    .groupBy("registration_date")
    .agg(F.count("*").alias("user_count"))
    .orderBy(F.desc("user_count"))
)
user_metrics.write.mode("overwrite").format("delta").saveAsTable("aaa.gold.user_metrics")

# Maintenance: Run the OPTIMIZE command on all three resulting Gold tables.
#the task explicitly says to do that, although OPTIMIZE after ZORDER BY revert the optimization of z-ordering
spark.sql("OPTIMIZE aaa.gold.taxi_metrics")
spark.sql("OPTIMIZE aaa.gold.driver_metrics")
spark.sql("OPTIMIZE aaa.gold.user_metrics")


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

# Data Enrichment

In [0]:
#it's nearly impossible to generate driver's id again for another table so that there are some identical to those in the silver.drivers drivers' ids , so i use the existing ones, but i redistribute them through the taxi trips with some skewness

ids_df = (spark.table("aaa.silver.drivers")
    .select("driver_id").distinct()
    .agg(F.sort_array(F.collect_list("driver_id")).alias("ids"))
    .withColumn("n", F.size("ids")))

taxi = (spark.table("aaa.silver.taxi")
    .crossJoin(F.broadcast(ids_df))
    .withColumn("driver_id", F.col("ids")[F.floor(F.pow(F.rand(42), 2) * F.col("n")).cast("int")])
    .drop("ids", "n"))
    
drivers = spark.table("aaa.silver.drivers").withColumnRenamed("ingest_time", "driver_ingest_time")
enriched_trips = taxi.join(drivers, on="driver_id", how="left")

(enriched_trips.write.mode("overwrite").option("overwriteSchema", "true")
    .format("delta").saveAsTable("aaa.gold.enriched_trips"))